

# Brute Force Warm-Up: Cracking Passwords
### OPIM 5641 - Business Decision Modeling · Module 2

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5641-notebooks/blob/main/3_BruteForce/Password_Cracking_Warmup.ipynb)

*Run me top to bottom - **Runtime → Run all**. Nothing to install, nothing to upload.*

🔴
<!-- 🎙 DAVE TALKING POINTS (invisible when rendered - double-click this cell to read):
- OPEN with the garage door opener: the 1980s toy that sweeps the WHOLE frequency spectrum and opens anybody's door. That is brute force.
- THE PIVOT from Module 1: "Is there a DISTRIBUTION of values for your password, or is there ONE password?" ONE. That is the difference between Monte Carlo and optimization.
- Password cracking = AAA, AAAB, AAAC ... until you hit apple.
- Your own quiz question, finally answered on camera: how many tries to reach 'apple'?
- CLOSE: "same nested loops next video - but now with constraints and an objective."
-->


Before we optimize anything, let's warm up on the purest brute force there is: guessing a password.

You've seen me tell this story in class. There's an **1980s toy garage-door opener** that sweeps the entire frequency spectrum - every wavelength, one after another - and it will open *anybody's* garage door. It isn't clever. It doesn't need to be. It just tries everything until something works. That is brute force in one image.

And here's the pivot from Module 1. In Monte Carlo we asked: what's the *distribution* of outcomes? Now ask yourself - **is there a distribution of values for your computer password, or is there one password?** One. Exactly one. That's the difference between simulation and optimization: simulation characterizes uncertainty, optimization hunts for **the** answer.

🔷 **The nugget:** brute force is a triple-nested (or N-nested) for loop that refuses to be smart. It always finds the answer - the only question is whether you'll still be alive when it does.

## Part 1 - a 5-digit PIN

Start small. Your phone PIN is **5 digits**, each one 0 through 9. How many possible PINs is that?

$$10 \times 10 \times 10 \times 10 \times 10 = 10^5 = 100{,}000$$

Five decisions, ten choices each - so **five nested for loops**, one per digit. That's the same shape as the chairs/desks/tables loops we're about to write for real optimization, just with digits instead of products.

In [ ]:
# the secret we are trying to crack (in real life you would not know this!)
secret_pin = '12345'

attempts = 0        # count every guess we make - lives OUTSIDE the loops
found = None        # the answer, once we trip over it

# five digits = five nested for loops
for d1 in range(0, 10):
  for d2 in range(0, 10):
    for d3 in range(0, 10):
      for d4 in range(0, 10):
        for d5 in range(0, 10):
          attempts = attempts + 1                      # another guess burned
          guess = str(d1) + str(d2) + str(d3) + str(d4) + str(d5)
          if guess == secret_pin:                      # did we get it?
            found = guess

print('Cracked:', found)
print('Attempts:', f'{attempts:,}', 'out of', f'{10**5:,}', 'possible PINs')

Notice we let the loop run to the bitter end - all 100,000 guesses - even after we found it. Let's be a little kinder to ourselves and **stop the moment we succeed**, then look at *where* in the enumeration the answer was hiding.

In [ ]:
# same search, but we stop as soon as we crack it
secret_pin = '12345'
attempts = 0
found = None

for d1 in range(0, 10):
  for d2 in range(0, 10):
    for d3 in range(0, 10):
      for d4 in range(0, 10):
        for d5 in range(0, 10):
          attempts = attempts + 1
          guess = str(d1) + str(d2) + str(d3) + str(d4) + str(d5)
          if guess == secret_pin:
            found = guess
            break                    # bail out of the innermost loop
        if found: break              # ... and each loop above it
      if found: break
    if found: break
  if found: break

print('Cracked:', found, 'after', f'{attempts:,}', 'attempts')
print('That is', f'{attempts/10**5:.1%}', 'of the way through all the possibilities')

Look at that number: we cracked `12345` on attempt **12,346**.

That's no coincidence - it's the whole idea of enumeration. Counting `00000, 00001, 00002, ...` in order IS counting in base 10, so a PIN's numeric value *is* its position in the search. (Why 12,346 and not 12,345? Because `00000` was guess number one. Same off-by-one that bites you with `np.arange` - counting from zero follows you everywhere.)

And that tells you something uncomfortable: **`00000` falls on the first guess and `99999` falls on the last.** Your "random" PIN is only as safe as how deep into the enumeration it happens to sit.

**Caution:** breaking out of five nested loops takes five `break` statements - one per level - because `break` only escapes the loop it's standing in. It's ugly. Keep it in mind when we get to optimization, where we DON'T break early: we want to score every feasible plan, not stop at the first one.

## Part 2 - cracking `apple`

Now letters. Same idea: the password is **5 letters**, each one `a` through `z`. That's

$$26 \times 26 \times 26 \times 26 \times 26 = 26^5 = 11{,}881{,}376$$

Almost **twelve million** combinations - from `aaaaa` all the way to `zzzzz`. And five letters means, you guessed it, **five nested for loops**.

This is the quiz question I always threaten in class and never actually ask: *if you loop a-to-z, a-to-z, a-to-z, a-to-z, a-to-z, when do you hit `apple`?* Today we find out.

In [ ]:
import string  # gives us the alphabet without typing 26 letters
import time      # so we can time the crack

letters = string.ascii_lowercase   # 'abcdefghijklmnopqrstuvwxyz'
print(letters, '->', len(letters), 'letters')
print('total 5-letter combinations:', f'{len(letters)**5:,}')

In [ ]:
# crack 'apple' - five letters, five nested loops
secret = 'apple'
attempts = 0
found = None
start = time.time()

for c1 in letters:
  for c2 in letters:
    for c3 in letters:
      for c4 in letters:
        for c5 in letters:
          attempts = attempts + 1
          guess = c1 + c2 + c3 + c4 + c5
          if guess == secret:
            found = guess
            break
        if found: break
      if found: break
    if found: break
  if found: break

elapsed = time.time() - start
print('Cracked:', found, 'after', f'{attempts:,}', 'attempts')
print('That is', f'{attempts/26**5:.1%}', 'of the search space')
print('Took', round(elapsed, 2), 'seconds')

**274,071 attempts** - and we got off easy. `apple` starts with `a`, the very first letter of our alphabet, so it sits only **2.3%** into the enumeration. Pure luck.

Change the secret to `zebra` and re-run: that one hides about **96.8%** of the way through, so you grind out roughly 11.5 million guesses instead of a quarter million. Same code, same password length, about **42 times** the work - purely because of which letters happen to be in the word.

**On your own:** change `secret` to `zebra` and time it. Then try `aaaaa` (the very first guess) and `zzzzz` (the very last). *That spread between best case and worst case is exactly why we judge an algorithm by its WORST case.*

🔴
<!-- 🎙 DAVE TALKING POINTS (invisible when rendered - double-click this cell to read):
- OPEN: "This is why IT makes you use a capital letter and a number."
- Walk the table: 26 -> 52 is 32x, 26 -> 62 is 77x. Same 5 characters!
- Then LENGTH: 5 chars -> 8 chars is 17,000x. Length beats complexity.
- We measure the rate ourselves - engineering approach, not hand-waving.
- THE HANDOFF: "next video, same nested loops - but now the combinations are BUSINESS PLANS, and we are not looking for the one right answer, we are looking for the BEST one."
-->


## Part 3 - why IT makes you use capitals, numbers, and symbols

Every password rule you've ever grumbled about exists to make this search bigger. Watch what happens to our 5-character password when we widen the alphabet:

- lowercase only: **26** choices per character
- lowercase + UPPERCASE: **52**
- lowercase + UPPERCASE + digits: **62**

We haven't made the password any *longer* - just widened what each slot can hold. Let's put real numbers on it, using the guess rate we just measured ourselves rather than making one up.

In [ ]:
# measure our own guess rate from the 'apple' crack above - honest engineering, no hand-waving
rate = attempts / elapsed
print('our machine guesses about', f'{rate:,.0f}', 'passwords per second')

In [ ]:
# how much bigger does the haystack get?
import pandas as pd

charsets = {
    'lowercase (26)': 26,
    'lowercase + UPPERCASE (52)': 52,
    'lowercase + UPPERCASE + digits (62)': 62,
}

rows = []
for name, n in charsets.items():
    combos = n**5                                  # 5-character password
    rows.append({
        'character set': name,
        'combinations': f'{combos:,}',
        'vs. lowercase': f'{combos/26**5:.0f}x',
        'seconds to try all': f'{combos/rate:,.0f}',
    })

pd.DataFrame(rows)

Same five characters - and adding capitals alone multiplies the work by **32**, while capitals plus digits multiplies it by **77**.

Now the bigger lever. Keep the alphabet at lowercase-only, but make the password **8 characters** instead of 5:

In [ ]:
# length beats complexity - every extra character MULTIPLIES the space by 26
for length in [5, 6, 7, 8]:
    combos = 26**length
    seconds = combos/rate
    if seconds < 3600:
        pretty = f'{seconds:,.1f} seconds'
    else:
        pretty = f'{seconds/3600:,.1f} hours'
    print(f'{length} lowercase characters: {combos:>18,} combinations  ~ {pretty}')

Five characters to eight - just three more letters - and the search goes from **about five seconds** to **most of a day**: a **17,576x** jump ($26^3$). That's why "make it longer" beats "add a squiggle" every single time.

**Remember:** these are toy numbers on a laptop running plain Python. A serious cracking rig does billions of guesses per second and doesn't start at `aaaaa` - it starts with a dictionary of real words and common substitutions, because humans pick `apple` and `P@ssw0rd`, not `xqjvz`. Brute force is the *dumbest* possible attack and it's still this fast. (This is a lesson in why algorithms matter, not a how-to. Use a password manager.)

## Part 4 - so what does this have to do with optimization?

Everything. Look at what we just did:

1. We listed every value each slot could take (`a` to `z`).
2. We built every combination with **nested for loops** - one loop per decision.
3. We checked each combination against a test.

Now swap the words and you have the optimization problem waiting in the next notebook:

| Password cracking | Optimization (Veerman Furniture) |
|---|---|
| 5 letter slots | 3 products: chairs, desks, tables |
| each slot: `a` to `z` | each product: 0 to its demand limit |
| 5 nested for loops | 3 nested for loops |
| test: does the guess match? | test: does the plan fit in our hours? (**feasible**) |
| exactly ONE right answer | MANY feasible answers - we want the **best** one |
| stop when found | keep going, remember the champion |

That last row is the only real difference, and it's the reason optimization has an **objective function**. A password has one right answer, so you stop. A business has thousands of workable plans, so you can't stop - you have to score every one and hang onto the winner.

**Caution:** and if you're already thinking *"twelve million guesses for a five-letter word... that's going to be a problem for a real business"* - you're exactly right. Hold that thought for about two videos.

## Bottom line

- **Brute force = try everything.** N decisions means N nested for loops. It always works, and it needs zero cleverness.
- Where the answer *sits* in the enumeration decides your luck: `apple` cost us 274,070 guesses, `zebra` would cost 11.5 million.
- Widening the alphabet multiplies the work (**32x** for capitals, **77x** with digits); adding **length** multiplies it far harder (**17,000x** from 5 to 8 characters).
- Optimization is this exact loop with two changes: combinations get filtered by **constraints**, and survivors get scored by an **objective**.

**On your own:**
1. Crack a 4-character password over lowercase + digits (36 choices). How many combinations, and how long does it actually take?
2. Rewrite the `apple` crack using `itertools.product(letters, repeat=5)` instead of five nested loops. Same answer, one line - but make sure you can still write the loops by hand, because *that's* the version the weekly check asks for.
3. Our loops always start at `aaaaa`. Real crackers start with a dictionary of common words. Roughly where in our enumeration would `admin` fall - and what does that tell you about picking real words as passwords?

*Next up: the same nested loops, but the combinations are production plans and we're hunting for the most profitable one.*